In [ ]:
# Example 1 — sigstore CLI: `python -m sigstore --version` reports the installed package version
import subprocess, sys
import sigstore

print("sigstore", sigstore.__version__)
proc = subprocess.run(
    [sys.executable, "-m", "sigstore", "--version"],
    capture_output=True, text=True, timeout=120,
)
assert proc.returncode == 0, proc.stderr
assert sigstore.__version__ in proc.stdout
print(proc.stdout.strip())

In [ ]:
# Example 2 — sigstore.hashes.Hashed: wrap a precomputed SHA-256 digest
import hashlib
from sigstore.hashes import HashAlgorithm, Hashed

digest = hashlib.sha256(b"my artifact").digest()
hashed = Hashed(algorithm=HashAlgorithm.SHA2_256, digest=digest)
assert hashed.algorithm == HashAlgorithm.SHA2_256
assert hashed.digest == digest
print("sha2-256:", hashed.digest.hex())

In [ ]:
# Example 3 — in-toto Statement construction via sigstore.dsse (offline)
import hashlib, json
from sigstore.dsse import StatementBuilder, Subject

sha256 = hashlib.sha256(b"release.tar.gz contents").hexdigest()
statement = StatementBuilder(
    subjects=[Subject(name="release.tar.gz", digest={"sha256": sha256})],
    predicate_type="https://slsa.dev/provenance/v1",
    predicate={"buildType": "https://example.com/manual-build/v1"},
).build()

# sigstore exposes no public accessor for the serialized statement; `_contents` is the private field holding it.
payload = json.loads(statement._contents)
assert payload["_type"] == "https://in-toto.io/Statement/v1"
assert payload["subject"][0]["digest"]["sha256"] == sha256
assert payload["predicateType"] == "https://slsa.dev/provenance/v1"
print(json.dumps(payload["subject"], indent=2))


In [ ]:
# Example 4 — sigstore.verify.policy: evaluate an Identity policy against a locally
# minted certificate (offline; SAN email + the raw-utf8 Fulcio issuer extension
# 1.3.6.1.4.1.57264.1.1 that Identity/OIDCIssuer checks)
import datetime
from cryptography import x509
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec
from sigstore.errors import VerificationError
from sigstore.verify.policy import Identity

key = ec.generate_private_key(ec.SECP256R1())
now = datetime.datetime.now(datetime.timezone.utc)
cert = (
    x509.CertificateBuilder()
    .subject_name(x509.Name([])).issuer_name(x509.Name([]))
    .public_key(key.public_key()).serial_number(x509.random_serial_number())
    .not_valid_before(now).not_valid_after(now + datetime.timedelta(minutes=10))
    .add_extension(x509.SubjectAlternativeName(
        [x509.RFC822Name("release@example.com")]), critical=False)
    .add_extension(x509.UnrecognizedExtension(
        x509.ObjectIdentifier("1.3.6.1.4.1.57264.1.1"), b"https://ci.example.com"),
        critical=False)
    .sign(key, hashes.SHA256())
)
Identity(identity="release@example.com", issuer="https://ci.example.com").verify(cert)
try:
    Identity(identity="someone-else@example.com", issuer="https://ci.example.com").verify(cert)
    raise AssertionError("wrong identity unexpectedly verified")
except VerificationError:
    print("OK — policy accepts the right identity and rejects the wrong one")


In [ ]:
# Example 5 — sigstore.oidc.IdentityToken: claim parsing of a supplied token (offline)
# A crafted unsigned JWT exercises only the local claim-validation path — the
# real OIDC/Fulcio flow needs network and is out of scope for this notebook.
import time
import jwt  # pyjwt, a declared sigstore runtime dep
from sigstore.oidc import IdentityToken

now = int(time.time())
claims = {
    "aud": "sigstore", "iss": "https://token.example.com",
    "sub": "test@example.com", "iat": now, "nbf": now, "exp": now + 600,
}
token = IdentityToken(jwt.encode(claims, "test-key", algorithm="HS256"))
assert token.identity == "test@example.com"
assert token.issuer == "https://token.example.com"
assert token.in_validity_period()
print("identity:", token.identity, "| issuer:", token.issuer)
